# GHSL Heatwave Exposure by Degree of Urbanisation — Brazil 2015

Combines three datasets to estimate how many people in each urbanisation category
are exposed to heatwaves:

| Dataset | Source | Role |
|---|---|---|
| ERA5 Daily TX | `ECMWF/ERA5/DAILY` | Heatwave days per pixel |
| GHS-SMOD R2023A | `projects/sat-io/open-datasets/GHS/GHS_SMOD` | Degree of urbanisation (1 km) |
| GHS-POP R2023A | `projects/sat-io/open-datasets/GHS/GHS_POP` | Population count (1 km) |

**Heatwave definition:** ≥ 3 consecutive days where daily TX > TX95p threshold  
**Threshold:** 95th percentile of ERA5 daily TX over the 1981–2010 baseline (annual, per pixel)  
**Analysis year:** 2015  
**Country:** Brazil

> **Note on threshold method:** This uses an *annual* 95th percentile (across all baseline days)
> rather than the ETCCDI calendar-day approach. For Brazil's predominantly tropical climate
> this is a reasonable first-pass approximation. A calendar-day threshold would require
> more complex GEE operations but could be added in a future iteration.

**Pipeline:**
```
ERA5 1981–2010  →  TX95p threshold image (~27 km)
ERA5 2015       →  daily exceedance  →  3-day consecutive run  →  heatwave days image
                →  bilinear resample to 1 km
GHS-SMOD 2015   →  urbanisation class per pixel (1 km)
GHS-POP  2015   →  population per pixel (1 km)
Combine         →  population grouped by SMOD class × heatwave exposure tier
```

## 0. Configuration
All editable parameters live here.

In [ ]:
# ── Country & epoch ───────────────────────────────────────────────────────────
COUNTRY_NAME     = "Brazil"
ANALYSIS_YEAR    = 2015         # year to count heatwave days for
SMOD_POP_YEAR    = 2015         # GHS-SMOD / GHS-POP epoch (5-yr steps: 1975–2020)

# ── ERA5 baseline ─────────────────────────────────────────────────────────────
BASELINE_START   = 1981
BASELINE_END     = 2010         # ETCCDI standard 30-year baseline

# ── Heatwave definition ───────────────────────────────────────────────────────
PERCENTILE       = 95           # TX percentile for threshold
MIN_CONSECUTIVE  = 3            # minimum consecutive exceedance days

# ── Resolution ───────────────────────────────────────────────────────────────
TARGET_SCALE_M   = 1000         # 1 km — SMOD / POP native resolution
ERA5_SCALE_M     = 27830        # ERA5/DAILY native (~0.25°)

# ── GEE ───────────────────────────────────────────────────────────────────────
GEE_PROJECT      = "tl-cities"
ERA5_DAILY_ID    = "ECMWF/ERA5/DAILY"
ERA5_TX_BAND     = "maximum_2m_air_temperature"    # Kelvin
GHS_SMOD_ID      = "projects/sat-io/open-datasets/GHS/GHS_SMOD"
GHS_POP_ID       = "projects/sat-io/open-datasets/GHS/GHS_POP"
BOUNDARY_ID      = "FAO/GAUL/2015/level0"

# ── Outputs ───────────────────────────────────────────────────────────────────
DRIVE_FOLDER     = "GHSL_Analysis"
OUTPUT_PREFIX    = f"brazil_{ANALYSIS_YEAR}"

import pathlib
OUTPUT_DIR = pathlib.Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Configuration loaded.")
print(f"  Country  : {COUNTRY_NAME}")
print(f"  Year     : {ANALYSIS_YEAR}")
print(f"  Baseline : {BASELINE_START}–{BASELINE_END}")
print(f"  Heatwave : ≥{MIN_CONSECUTIVE} consecutive days above TX{PERCENTILE}p")
print(f"  Metric   : person-heatwave-days = Σ(hw_days × population) per SMOD class")
print(f"  Scale    : {TARGET_SCALE_M} m (analysis), {ERA5_SCALE_M} m (ERA5 native)")

## 1. Imports & GEE Initialisation

In [6]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05)

try:
    ee.Initialize(project=GEE_PROJECT)
    print(f"GEE initialized (project: {GEE_PROJECT})")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print(f"GEE authenticated and initialized (project: {GEE_PROJECT})")

GEE initialized (project: tl-cities)


## 2. Country Boundary

In [7]:
brazil_fc = (
    ee.FeatureCollection(BOUNDARY_ID)
      .filter(ee.Filter.eq("ADM0_NAME", COUNTRY_NAME))
)
brazil = brazil_fc.geometry()

area_km2 = brazil.area(maxError=1000).divide(1e6).getInfo()
bounds   = brazil.bounds().getInfo()["coordinates"][0]
lon_min  = min(c[0] for c in bounds)
lon_max  = max(c[0] for c in bounds)
lat_min  = min(c[1] for c in bounds)
lat_max  = max(c[1] for c in bounds)

print(f"{COUNTRY_NAME} boundary loaded")
print(f"  Area    : {area_km2:,.0f} km²")
print(f"  Lon     : {lon_min:.2f} → {lon_max:.2f}")
print(f"  Lat     : {lat_min:.2f} → {lat_max:.2f}")

Brazil boundary loaded
  Area    : 8,518,291 km²
  Lon     : -73.99 → -28.85
  Lat     : -33.75 → 5.26


## 3. ERA5 TX95p Threshold (1981–2010 Baseline)

Computes the per-pixel 95th percentile of daily TX across all days in the
1981–2010 baseline period using GEE's server-side `percentile` reducer.

This is a lazy operation — GEE evaluates it when the image is consumed
(map tiles, export, or getInfo). No data is downloaded here.

In [ ]:
baseline_col = (
    ee.ImageCollection(ERA5_DAILY_ID)
      .filterDate(f"{BASELINE_START}-01-01", f"{BASELINE_END + 1}-01-01")
      .filterBounds(brazil)
      .select(ERA5_TX_BAND)
)

n_baseline = baseline_col.size().getInfo()
print(f"Baseline collection: {BASELINE_START}–{BASELINE_END}")
print(f"  Images: {n_baseline:,} daily images")

# Per-pixel TX95p threshold (Kelvin) — server-side lazy computation.
# Not evaluated here: computing percentile across ~11k images × full Brazil
# synchronously would hang. GEE evaluates this when tiles render or
# the analysis reducers run (Sections 7 & 8).
threshold_k = (
    baseline_col
      .reduce(ee.Reducer.percentile([PERCENTILE]))
      .rename("tx95p")
      .clip(brazil)
)
print(f"TX{PERCENTILE}p threshold image defined (lazy).")
print("Verify the spatial pattern via the TX95p layer in Section 7's map.")

## 4. Heatwave Days in 2015

### Method
1. For each day in 2015: `exc[d] = 1` if TX > TX95p, else 0.
2. Compute the 3-day **forward** sum `S[d] = exc[d] + exc[d+1] + exc[d+2]`.
3. Day `d` is a **heatwave day** if `exc[d] = 1` AND any of:
   - `S[d] ≥ 3` — d starts a qualifying run
   - `S[d−1] ≥ 3` — d is the middle day of a run
   - `S[d−2] ≥ 3` — d is the end day of a run
4. Sum heatwave day flags across all days → total heatwave days per pixel.

The computation graph is built client-side in a Python loop (365 iterations);
all image operations execute lazily on GEE servers.

In [ ]:
# ── Load 2015 ERA5 TX ─────────────────────────────────────────────────────────
era5_2015 = (
    ee.ImageCollection(ERA5_DAILY_ID)
      .filterDate(f"{ANALYSIS_YEAR}-01-01", f"{ANALYSIS_YEAR + 1}-01-01")
      .filterBounds(brazil)
      .select(ERA5_TX_BAND)
      .sort("system:time_start")
)
n_days = era5_2015.size().getInfo()
print(f"ERA5 {ANALYSIS_YEAR}: {n_days} daily images")

# ── Binary exceedance per day ─────────────────────────────────────────────────
def mark_exceedance(img):
    return (img.gt(threshold_k)
              .rename("exc")
              .copyProperties(img, ["system:time_start"]))

exc_col  = era5_2015.map(mark_exceedance)
exc_list = exc_col.toList(n_days)

# ── Pre-cache exceedance image references (lazy, client-side) ─────────────────
exc_imgs = [ee.Image(exc_list.get(d)).rename("e") for d in range(n_days)]
zero_img = ee.Image.constant(0).rename("e")

def safe_exc(d):
    """Return exceedance image at index d, or zero image if out of bounds."""
    return exc_imgs[d] if 0 <= d < n_days else zero_img

# ── Build heatwave day images (Python loop → GEE computation graph) ───────────
print("Building heatwave computation graph (365 iterations)...", end=" ")
hw_images = []
for d in range(n_days):
    e0 = safe_exc(d)

    # 3-day forward sum starting at d, d-1, d-2
    s_d   = safe_exc(d  ).add(safe_exc(d+1)).add(safe_exc(d+2))
    s_dm1 = safe_exc(d-1).add(safe_exc(d  )).add(safe_exc(d+1))
    s_dm2 = safe_exc(d-2).add(safe_exc(d-1)).add(safe_exc(d  ))

    in_run = (
        s_d  .gte(MIN_CONSECUTIVE)
        .Or(s_dm1.gte(MIN_CONSECUTIVE))
        .Or(s_dm2.gte(MIN_CONSECUTIVE))
    )
    hw_images.append(e0.And(in_run).rename("hw_day"))

# ── Sum across the year → total heatwave days per pixel ──────────────────────
hw_days_era5 = (
    ee.ImageCollection(hw_images)
      .sum()
      .rename("hw_days")
      .clip(brazil)
)
print("done.")
print(f"Heatwave days image ready (≥{MIN_CONSECUTIVE} consecutive days above TX{PERCENTILE}p, {ANALYSIS_YEAR}).")

## 5. Resample ERA5 Heatwave Days to 1 km

Bilinear resampling upscales from ~27 km to 1 km.  
This does **not** add spatial information — it smooths the ERA5 signal to align
with the SMOD/POP grid.

In [ ]:
hw_days_1km = (
    hw_days_era5
      .resample("bilinear")
      .reproject(crs="EPSG:4326", scale=TARGET_SCALE_M)
      .clip(brazil)
)
print(f"Heatwave days resampled: {ERA5_SCALE_M} m → {TARGET_SCALE_M} m (bilinear, EPSG:4326)")

## 6. GHS-SMOD and GHS-POP (2015 Epoch)

Both collections contain multi-epoch images. The 2015 epoch is selected
by filtering to a date window centred on 2015.

In [ ]:
import datetime

smod_col = ee.ImageCollection(GHS_SMOD_ID)
pop_col  = ee.ImageCollection(GHS_POP_ID)

# ── Print all image IDs so the epoch filter can be verified ──────────────────
smod_ids = smod_col.aggregate_array("system:index").getInfo()
print(f"GHS-SMOD collection ({len(smod_ids)} images):")
for i in smod_ids:
    print(f"  {i}")

# ── Select the epoch matching SMOD_POP_YEAR by index string ──────────────────
# sat-io collections encode the epoch year in system:index, e.g.
# "GHS_SMOD_E2015_GLOBE_R2023A_54009_1000_V1_0"
smod_raw = smod_col.filter(
    ee.Filter.stringContains("system:index", str(SMOD_POP_YEAR))
).first()
pop_raw  = pop_col.filter(
    ee.Filter.stringContains("system:index", str(SMOD_POP_YEAR))
).first()

smod_band = smod_raw.bandNames().getInfo()[0]
pop_band  = pop_raw.bandNames().getInfo()[0]
print(f"\nSMOD {SMOD_POP_YEAR} — image id : {smod_raw.get('system:index').getInfo()}")
print(f"                  band   : {smod_band}")
print(f"POP  {SMOD_POP_YEAR} — image id : {pop_raw.get('system:index').getInfo()}")
print(f"                  band   : {pop_band}")

# ── Reproject both to 1 km EPSG:4326, clipped to country ─────────────────────
smod_img = (
    smod_raw.select([smod_band]).rename("smod")
            .reproject(crs="EPSG:4326", scale=TARGET_SCALE_M)
            .clip(brazil)
)
pop_img = (
    pop_raw.select([pop_band]).rename("population")
           .reproject(crs="EPSG:4326", scale=TARGET_SCALE_M)
           .clip(brazil)
)

# ── Person-heatwave-days: hw_days × population per pixel ─────────────────────
person_hw_days = hw_days_1km.multiply(pop_img).rename("person_hw_days")
print("\nAll images defined — smod_img, pop_img, person_hw_days ready.")

## 7. Person-Heatwave-Days by SMOD Class

For each SMOD class, sum `hw_days × population` across all pixels in that class.
Also sum raw population for normalisation.

7 reductions run in parallel (one `getInfo` per SMOD class).
**Expect 5–15 minutes** depending on GEE server load.

In [ ]:
SMOD_CLASSES = {
    30: "Urban Centre",
    23: "Dense Urban Cluster",
    22: "Semi-dense Urban Cluster",
    21: "Suburban / Peri-urban",
    13: "Rural Cluster",
    12: "Low Density Rural",
    11: "Very Low Density Rural",
    # 10 = Water — excluded from population analysis
}
SMOD_ORDER = [30, 23, 22, 21, 13, 12, 11]  # urban → rural

In [ ]:
def stats_for_smod(smod_code, _smod_img, _pop_img, _person_hw_days, _brazil):
    """Sum population and person-heatwave-days for one SMOD class."""
    mask = _smod_img.eq(smod_code)
    result = (
        _pop_img.rename("population")
                .addBands(_person_hw_days.rename("person_hw_days"))
                .updateMask(mask)
                .reduceRegion(
                    reducer   = ee.Reducer.sum(),
                    geometry  = _brazil,
                    scale     = TARGET_SCALE_M,
                    maxPixels = 1e12,
                    bestEffort= True,
                )
                .getInfo()
    )
    return {
        "smod_code":      smod_code,
        "smod_label":     SMOD_CLASSES[smod_code],
        "population":     result.get("population")     or 0.0,
        "person_hw_days": result.get("person_hw_days") or 0.0,
    }


print(f"Running {len(SMOD_ORDER)} reductions in parallel (one per SMOD class)...")

rows = []
with ThreadPoolExecutor(max_workers=7) as pool:
    futures = {
        pool.submit(stats_for_smod, sc, smod_img, pop_img, person_hw_days, brazil): sc
        for sc in SMOD_ORDER
    }
    for future in as_completed(futures):
        row = future.result()
        rows.append(row)
        print(
            f"  {row['smod_label']:28s}  "
            f"pop={row['population']/1e6:6.2f}M  "
            f"person-hw-days={row['person_hw_days']:,.0f}",
            end="\r"
        )

print(f"\nDone.")

df = (
    pd.DataFrame(rows)
      .sort_values("smod_code", ascending=False)
      .reset_index(drop=True)
)
df["mean_hw_days"] = df["person_hw_days"] / df["population"].replace(0, float("nan"))

csv_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_person_hw_days_by_smod.csv"
df.to_csv(csv_path, index=False)
print(f"Results saved → {csv_path}")

## 8. Results & Visualisations

In [ ]:
smod_label_order = [SMOD_CLASSES[c] for c in SMOD_ORDER]
df_plot = df.set_index("smod_label").reindex(smod_label_order)

summary = pd.DataFrame({
    "Population":            df_plot["population"].map(lambda x: f"{x/1e6:.2f}M"),
    "Person-heatwave-days":  df_plot["person_hw_days"].map(lambda x: f"{x:,.0f}"),
    "Mean hw days / person": df_plot["mean_hw_days"].map(lambda x: f"{x:.1f}"),
})
summary.index.name = "SMOD class (urban → rural)"

print(f"{COUNTRY_NAME} {ANALYSIS_YEAR} — Person-Heatwave-Days by Degree of Urbanisation")
print(f"(heatwave = ≥{MIN_CONSECUTIVE} consecutive days above TX{PERCENTILE}p)\n")
display(summary)

national_phd = df["person_hw_days"].sum()
national_pop = df["population"].sum()
print(f"\nNational total  person-heatwave-days : {national_phd:,.0f}")
print(f"National mean   hw days per person    : {national_phd / national_pop:.1f}")

In [ ]:
# ── Figure 1: Total person-heatwave-days by SMOD class ───────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#c0392b", "#e74c3c", "#e67e22", "#f39c12", "#27ae60", "#2ecc71", "#85c1e9"]
bars = ax.bar(
    smod_label_order,
    df_plot["person_hw_days"].values / 1e6,
    color=colors, edgecolor="white", linewidth=0.4
)
ax.bar_label(bars, fmt="%.2f M", padding=4, fontsize=9)
ax.set_xlabel("Degree of Urbanisation (GHS-SMOD)", fontsize=11)
ax.set_ylabel("Person-heatwave-days (millions)", fontsize=11)
ax.set_title(
    f"Total Person-Heatwave-Days by Degree of Urbanisation\n"
    f"{COUNTRY_NAME}  ·  {ANALYSIS_YEAR}  ·  TX{PERCENTILE}p  ·  ≥{MIN_CONSECUTIVE} consecutive days",
    fontsize=12
)
ax.tick_params(axis="x", rotation=35)
sns.despine()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{OUTPUT_PREFIX}_person_hw_days_bar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figure 2: Population-weighted mean heatwave days per person ───────────────
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    smod_label_order,
    df_plot["mean_hw_days"].values,
    color=colors, edgecolor="white", linewidth=0.4
)
ax.bar_label(bars, fmt="%.1f d", padding=4, fontsize=9)
ax.set_xlabel("Degree of Urbanisation (GHS-SMOD)", fontsize=11)
ax.set_ylabel("Mean heatwave days per person", fontsize=11)
ax.set_title(
    f"Population-Weighted Mean Heatwave Days by Degree of Urbanisation\n"
    f"{COUNTRY_NAME}  ·  {ANALYSIS_YEAR}",
    fontsize=12
)
ax.tick_params(axis="x", rotation=35)
sns.despine()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"{OUTPUT_PREFIX}_mean_hw_days_bar.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Export Stacked Raster to Google Drive (Optional)

Exports a 4-band GeoTIFF at 1 km resolution for offline analysis:

| Band | Content |
|---|---|
| `hw_days` | Heatwave days in 2015 (integer) |
| `hw_tier` | Heatwave tier 0–4 (integer) |
| `smod` | GHS-SMOD class code |
| `population` | GHS-POP count per 1 km pixel |

In [ ]:
stacked = (
    hw_days_1km.rename("hw_days")
               .addBands(person_hw_days.rename("person_hw_days"))
               .addBands(smod_img.rename("smod"))
               .addBands(pop_img.rename("population"))
)

task = ee.batch.Export.image.toDrive(
    image           = stacked,
    description     = f"{OUTPUT_PREFIX}_ghsl_heatwave_smod_pop_1km",
    folder          = DRIVE_FOLDER,
    fileNamePrefix  = f"{OUTPUT_PREFIX}_ghsl_heatwave_smod_pop_1km",
    region          = brazil,
    scale           = TARGET_SCALE_M,
    crs             = "EPSG:4326",
    maxPixels       = 1e12,
    fileFormat      = "GeoTIFF",
)
task.start()
print(f"Export task submitted: {OUTPUT_PREFIX}_ghsl_heatwave_smod_pop_1km")
print(f"Output folder       : Google Drive / {DRIVE_FOLDER}")
print(f"Monitor             : https://code.earthengine.google.com/tasks")
print()
print("Bands: hw_days | person_hw_days | smod | population")
print("CRS  : EPSG:4326 @ 1 km")

## 10. Interactive Map (Optional)

Tile-based — GEE renders each tile on demand, so panning is slow while the
TX95p percentile computation runs server-side for the first time. Run this
after the analysis in Section 7 has completed so GEE has already cached
intermediate results.

In [ ]:
Map = geemap.Map()
Map.centerObject(brazil, zoom=4)

Map.addLayer(
    hw_days_1km.updateMask(hw_days_1km.gt(0)),
    {"min": 1, "max": 60,
     "palette": ["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"]},
    f"Heatwave days {ANALYSIS_YEAR}"
)
Map.addLayer(
    threshold_k.subtract(273.15),
    {"min": 25, "max": 42,
     "palette": ["#2166ac", "#f7f7f7", "#d73027"]},
    f"TX{PERCENTILE}p threshold (°C)",
    shown=False
)
Map.addLayer(
    smod_img,
    {"min": 10, "max": 30,
     "palette": ["#d4e6f1", "#85c1e9", "#2e86c1",
                 "#1a5276", "#f0e68c", "#f4d03f", "#e67e22", "#c0392b"]},
    "GHS-SMOD 2015",
    shown=False
)
Map.addLayer(
    ee.Image().paint(brazil_fc, 1, 2),
    {"palette": ["000000"]},
    f"{COUNTRY_NAME} boundary"
)
Map

## Notes & Limitations

1. **Threshold method**: Annual TX95p (all baseline days pooled). The ETCCDI standard
   uses a calendar-day percentile with a 5-day centred window, which accounts for
   seasonal variation. For tropical Brazil the difference is modest but non-negligible
   in the far south. A future iteration could implement the calendar-day approach
   by stacking DOY-grouped images on GEE.

2. **ERA5 resolution**: ~27 km. The bilinear resample to 1 km smooths the signal
   but does not add sub-27 km spatial detail. Urban heat island effects and
   fine-scale climate variability are not captured.

3. **`bestEffort=True`**: GEE may increase the computation scale automatically
   to avoid timeout. Check printed output for actual scales used.

4. **Population masking**: Pixels classified as Water (SMOD=10) are excluded.
   Very small pixel populations may be lost due to floating-point precision
   in the 1 km reprojection.

5. **Extending to other countries / years**: Change `COUNTRY_NAME`, `ANALYSIS_YEAR`,
   and `SMOD_POP_YEAR` in the configuration cell and re-run.